# Supplementary Figure 8 - Mouse pup method comparison

            This notebook is a cleaned, English-commented manuscript code file.

            **Provenance.** A100 `benchmarking_mouse_pup.ipynb` plus local mouse pup notebooks.

            The notebook keeps the original manuscript data paths where they were found. If a path points to
            `/data/taobo.hu` or `/mnt/taobo.hu`, run the notebook on the A100 server or adapt the path to the
            matching mounted Y-drive location.


In [ ]:
# Panel mapping:
# - Supplementary Figure 8a-d, shared setup. Imports plotting and table libraries used by the mouse pup method-comparison panels.
#

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
sns.set_theme(context="talk", style="white")


In [ ]:
# Panel mapping:
# - Supplementary Figure 8b-d, Squidpy helper. Builds AnnData and computes Squidpy neighborhood enrichment for the comparator panel.
#

import anndata as ad
import squidpy as sq


def dataframe_to_adata(df):
    """Convert a point table into a minimal AnnData object for neighborhood methods."""
    adata = ad.AnnData(X=np.zeros((df.shape[0], 1), dtype=float))
    adata.obs["celltype"] = pd.Categorical(df["celltype"].to_numpy())
    adata.obsm["spatial"] = df[["x", "y"]].to_numpy()
    adata.obs_names = [f"cell_{i}" for i in range(df.shape[0])]
    return adata


def compute_squidpy_ne(df, n_perms=1000, radius=None, n_neighs=None):
    """Run Squidpy neighborhood enrichment and return its z-score matrix."""
    adata = dataframe_to_adata(df)
    sq.gr.spatial_neighbors(adata, coord_type="generic", spatial_key="spatial", radius=radius, n_neighs=n_neighs)
    sq.gr.nhood_enrichment(adata, cluster_key="celltype", n_perms=n_perms)
    cats = adata.obs["celltype"].cat.categories
    z = adata.uns["celltype_nhood_enrichment"]["zscore"]
    return pd.DataFrame(z, index=cats, columns=cats)


In [ ]:
# Panel mapping:
# - Supplementary Figure 8a-d. Loads mouse pup Xenium data, writes the Cell-GPS/COSTE matrix panel, and writes the Squidpy comparator matrix source.
#

from sfplot import load_xenium_data, compute_cophenetic_distances_from_df, plot_cophenetic_heatmap

XENIUM_DIR = Path("Y:/long/10X_datasets/Xenium/Xenium_5K/Xenium_Prime_Mouse_Pup_FFPE_outs")
OUTPUT_DIR = Path("outputs/supp_fig_08_mouse_pup_method_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
adata = load_xenium_data(str(XENIUM_DIR), normalize=False)
adata.obs["x"] = adata.obsm["spatial"][:, 0]
adata.obs["y"] = adata.obsm["spatial"][:, 1]
cluster_key = "annotation" if "annotation" in adata.obs else "Cluster"
points = adata.obs[["x", "y", cluster_key]].rename(columns={cluster_key: "celltype"})

row_coph, _ = compute_cophenetic_distances_from_df(points, x_col="x", y_col="y", celltype_col="celltype")
row_coph.to_csv(OUTPUT_DIR / "COSTE_Mouse_Pup.csv")
plot_cophenetic_heatmap(row_coph, output_filename=str(OUTPUT_DIR / "COSTE_Mouse_Pup.pdf"), matrix_name="row_coph", sample="Mouse_Pup")

squidpy_z = compute_squidpy_ne(points, n_perms=1000)
squidpy_z.to_csv(OUTPUT_DIR / "Squidpy_Mouse_Pup.csv")
